# 06 - Fine-tune a small LLM with LoRA (Qwen3.5-0.8B, free Colab T4)

Goal: teach a 0.8B model to turn messy operator notes into our JSON defect record, **including the house rules for `action` that are never written in the prompt**. Then compare with the baselines from `05_baselines.py` (big Groq model, small model + prompt).

**Before running**
1. `Runtime > Change runtime type > T4 GPU`
2. Upload 4 files from `pillar4/` with the folder icon on the left: `notes_task.py`, `data/train.jsonl`, `data/val.jsonl`, `data/test.jsonl` (all at the root of the Colab file panel, no `data/` folder needed)
3. `Runtime > Run all`. About 20-30 min in total (install ~3 min, training ~10 min, GGUF export ~5-10 min)

**Tools**: [Unsloth](https://github.com/unslothai/unsloth) (faster, less memory) on top of Hugging Face `transformers`, `peft` (LoRA) and `trl` (SFTTrainer). If a cell fails because an API changed, compare with Unsloth's official Qwen3.5 notebooks (linked from their docs), they're kept up to date.

In [ ]:
%%capture
# Unsloth pins compatible versions of transformers / trl / peft for you
!pip install unsloth

## 1. Files and config

In [ ]:
import json, os, time
from pathlib import Path

for f in ["notes_task.py", "train.jsonl", "val.jsonl", "test.jsonl"]:
    assert Path(f).exists(), f"Upload {f} first (folder icon on the left)"

from notes_task import FIELDS, SYSTEM_PROMPT, parse_json, score, summarize

MODEL_NAME = "Qwen/Qwen3.5-0.8B"   # try "Qwen/Qwen3.5-2B" later (~5 GB for LoRA, still fits a T4)
MAX_SEQ = 512                      # notes + JSON are short, ~200 tokens
N_EVAL = 200                       # test notes used for before/after (all of them)

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines()]

train_rows, val_rows, test_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl"), read_jsonl("test.jsonl")
print(len(train_rows), len(val_rows), len(test_rows))
print(train_rows[0]["messages"][1]["content"], "->", train_rows[0]["messages"][2]["content"])

## 2. Load the model (16-bit, no QLoRA)

Unsloth discourages 4-bit QLoRA for Qwen3.5 (quantization error too high, see `02_quantization.py`). At 0.8B, 16-bit LoRA needs ~3 GB, a T4 has 15.

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bf16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
# Qwen3.5 is multimodal: what comes back can be a processor wrapping the text tokenizer
text_tok = getattr(tokenizer, "tokenizer", tokenizer)
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16 supported: {is_bf16_supported()} (T4: no, fp16 is used)")
print(f"memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Chat template: what the model actually sees

Fine-tuning data must be formatted with the model's own chat template (special tokens marking system / user / assistant turns). Using a different template at inference time (Ollama, vLLM...) is the #1 cause of "my fine-tuned model is worse after export".

In [ ]:
def to_text(row):
    return text_tok.apply_chat_template(row["messages"], tokenize=False)

def to_prompt(row):
    # system + user only, and the opening of the assistant turn: the model writes the rest
    return text_tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True,
                                        enable_thinking=False)

print(to_text(train_rows[0]))
print("=" * 60)
print(repr(to_prompt(train_rows[0])[-80:]))

INSTRUCTION_PART, RESPONSE_PART = "<|im_start|>user\n", "<|im_start|>assistant\n"
sample = to_text(train_rows[0])
assert INSTRUCTION_PART in sample and RESPONSE_PART in sample, (
    "The template doesn't use these markers. Look at the printed text and set INSTRUCTION_PART / RESPONSE_PART "
    "to the strings that open the user and assistant turns.")

## 4. Baseline: the model BEFORE fine-tuning

Same basic prompt as in `05_baselines.py`, greedy decoding, no JSON constraint.

In [ ]:
@torch.no_grad()
def evaluate(rows, batch_size=16, max_new_tokens=120):
    FastLanguageModel.for_inference(model)
    text_tok.padding_side = "left"          # left padding for batched generation
    if text_tok.pad_token is None:
        text_tok.pad_token = text_tok.eos_token
    scores, outputs, start = [], [], time.time()
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        enc = text_tok([to_prompt(r) for r in batch], return_tensors="pt", padding=True,
                       add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        texts = text_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for r, t in zip(batch, texts):
            outputs.append(t)
            scores.append(score(parse_json(t), r["label"]))
    s = summarize(scores)
    s["sec_per_note"] = round((time.time() - start) / len(rows), 3)
    return s, outputs

before, before_out = evaluate(test_rows[:N_EVAL])
print({k: before[k] for k in ["all", "valid_json", "action", "sec_per_note"]})
for r, o in list(zip(test_rows, before_out))[:3]:
    print("\nNOTE :", r["messages"][1]["content"], "\nMODEL:", o.strip()[:200], "\nTRUTH:", json.dumps(r["label"]))

## 5. Add LoRA adapters

`r=16` on all attention and MLP projections. Only the adapters are trained, look at the % of trainable parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",   # recompute activations: less memory, a bit slower
    random_state=3407,
)
model.print_trainable_parameters()

## 6. Train

- 1 epoch over 1,500 examples, effective batch 16 -> ~94 steps
- `train_on_responses_only`: the loss is computed only on the assistant's JSON, not on the system prompt or the note (we want the model to learn to *answer*, not to *write notes*)
- eval loss on `val.jsonl` every 25 steps: if it goes up while train loss goes down, you're overfitting

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

FastLanguageModel.for_training(model)
text_tok.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        output_dir="outputs",
        report_to="none",
        seed=3407,
        dataset_num_proc=1,
    ),
)
trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)

# sanity check: which tokens carry a loss? (-100 = ignored)
labels = trainer.train_dataset[0]["labels"]
kept = [t for t in labels if t != -100]
print("trained on:", repr(text_tok.decode(kept)))

In [ ]:
stats = trainer.train()
print(f"train time: {stats.metrics['train_runtime'] / 60:.1f} min, "
      f"peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(e["step"], e["loss"]) for e in log if "loss" in e]
ev = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]
plt.plot(*zip(*tr), label="train"); plt.plot(*zip(*ev), "o-", label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("LoRA fine-tuning"); plt.show()

## 7. After fine-tuning

In [ ]:
after, after_out = evaluate(test_rows[:N_EVAL])
print(f"{'field':<14}{'before':>8}{'after':>8}")
for k in ["all", "valid_json"] + FIELDS:
    print(f"{k:<14}{before[k]:>8.0%}{after[k]:>8.0%}")
print(f"\nsec/note on a T4 (batched): {after['sec_per_note']}")

errors = [(r, o) for r, o in zip(test_rows, after_out) if not score(parse_json(o), r["label"])["all"]]
print(f"\n{len(errors)} notes still wrong, first ones:")
for r, o in errors[:5]:
    print("NOTE :", r["messages"][1]["content"], "\nMODEL:", o.strip(), "\nTRUTH:", json.dumps(r["label"]), "\n")

json.dump({"qwen3.5-0.8b base (Colab, no constraint)": before, "qwen3.5-0.8b + LoRA (Colab)": after},
          open("finetune_results.json", "w"), indent=1)

## 8. Save: LoRA adapter + GGUF for Ollama

- the **adapter** alone is a few tens of MB: keep it, it's your fine-tune
- the **GGUF** is the merged model quantized to Q4_K_M, ready for Ollama / llama.cpp (`07_serve_and_compare.py`). Trained in 16-bit, deployed in 4-bit: check that the 4-bit version keeps the accuracy (07 measures it)

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")
print(sorted(os.listdir("gguf_model")))

In [ ]:
!zip -qr notes_finetune.zip gguf_model lora_adapter finetune_results.json
from google.colab import files
files.download("notes_finetune.zip")

## Where to go from here

- unzip `notes_finetune.zip` in `pillar4/outputs/` and run `python 07_serve_and_compare.py`
- experiments worth doing (change one thing at a time, compare with the table above):
  - 300 training examples instead of 1,500: how much data do you really need?
  - `r=4` vs `r=64`
  - `Qwen/Qwen3.5-2B`
  - 3 epochs: does val loss go up (overfitting)?